# 🧱 7. Layered / Combined Guardrails

In production systems, a single guardrail is often not enough.

Instead, multiple guardrails can be combined to provide **layered protection** at different stages of the agent workflow.

---

## 🔄 Layered Guardrail Architecture

```text
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response

---

## 🧩 Guardrail Layers

### 1️⃣ Deterministic Input Filter

Blocks clearly unsafe or unwanted requests using predefined rules and keywords.

```text
Input
  ↓
Keyword / Rule Check
  ↓
Allowed → Continue
Blocked → 🚫 Stop

### 2️⃣ PII Input Redaction

Detects sensitive information in the user's input and applies a predefined strategy such as redaction or masking.

```text
User Input
    ↓
PII Detection
    ↓
PII Found?
    ├── No  → Continue
    └── Yes → 🔒 Redact / Mask

### 3️⃣ Human Approval

Sensitive operations can be paused and sent to a human for approval.

```text
Sensitive Tool Call
    ↓
Human Approval
    ├── Approve → ✅ Continue
    └── Reject  → 🚫 Stop

### 4️⃣ PII Output Redaction

The agent's response can also be checked for sensitive information before it reaches the user.

```text
Agent Output
    ↓
PII Detection
    ↓
Sensitive Data?
    ├── No  → Continue
    └── Yes → 🔒 Redact / Mask

### 5️⃣ Model-Based Output Safety

A model-based guardrail evaluates the final output for unsafe or inappropriate content.

```text
Agent Output
    ↓
Safety Model
    ↓
Safe?
    ├── Yes → ✅ Return to User
    └── No  → 🚫 Block / Replace

---

## 🧱 Layered / Combined Guardrails Example

### Import Required Libraries

In [5]:
import os
from typing import Any

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    PIIMiddleware,
    HumanInTheLoopMiddleware,
    hook_config,
)
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver

### Environment Setup and Groq Model

In [6]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("✅ Groq model initialized!")

✅ Groq model initialized!


### Custom Middleware #1 — Before-Agent Guardrail

In [7]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail:
    Block requests containing banned keywords.

    Runs BEFORE the agent processes the request.
    This means blocked requests do not consume an LLM call.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()

        self.banned_keywords = [
            keyword.lower()
            for keyword in banned_keywords
        ]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        first_message = state["messages"][0]

        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:

            if keyword in content:

                print(
                    f"🚫 Blocked — keyword detected: '{keyword}'"
                )

                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing "
                            "inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }

        return None

### Custom Middleware #2 — After-Agent Guardrail

In [8]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail:

    Uses an LLM to evaluate the agent's output
    before it reaches the user.
    """

    def __init__(self, safety_model):
        super().__init__()

        self.safety_model = safety_model

    @hook_config()
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage):
            return None

        output = last_message.content

        safety_prompt = f"""
Evaluate the following AI-generated response for safety.

Reply with ONLY one word:

SAFE
or
UNSAFE

Response:
{output}
"""

        result = self.safety_model.invoke([
            {
                "role": "user",
                "content": safety_prompt
            }
        ])

        verdict = result.content.strip().upper()

        if "UNSAFE" in verdict:

            print("🚫 Unsafe output detected.")

            return {
                "messages": [{
                    "role": "assistant",
                    "content": (
                        "I'm sorry, but I cannot provide "
                        "that response."
                    )
                }]
            }

        print("✅ Output passed safety check.")

        return None

### Tools

In [9]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results for: {query}"


@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to}"


print("✅ Tools created!")

✅ Tools created!


### Build the 5-Layer Guardrail Agent

In [10]:
production_agent = create_agent(
    model=model,

    tools=[
        search_tool,
        send_email_tool
    ],

    middleware=[

        # Layer 1: Deterministic Input Filter
        ContentFilterMiddleware(
            banned_keywords=[
                "hack",
                "exploit",
                "malware",
                "jailbreak",
                "bypass"
            ]
        ),

        # Layer 2: PII Protection
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),

        # Layer 3: Human-in-the-Loop
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True,
                "search_tool": False
            }
        ),

        # Layer 4: Output PII Protection
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_output=True
        ),

        # Layer 5: Model-Based Output Safety
        SafetyGuardrailMiddleware(
            safety_model=model
        ),
    ],

    # Required for HumanInTheLoopMiddleware
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


### Test 1 — Safe Request

In [12]:
config = {
    "configurable": {
        "thread_id": "safe_request_001"
    }
}

result = production_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is machine learning?"
            }
        ]
    },
    config=config
)

print("✅ Safe Request:")
print(result["messages"][-1].content)

✅ Output passed safety check.
✅ Safe Request:
Machine learning (ML) is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from data and improve their performance on a specific task without being explicitly programmed. 

### Core Idea
- **Learning from Data:** Instead of writing explicit rules, you feed the system examples (data) and let it infer patterns.
- **Prediction/Decision Making:** Once trained, the model can make predictions, classify inputs, or take actions on new, unseen data.

### How It Works
1. **Collect Data:** Gather relevant examples (e.g., images, text, sensor readings).
2. **Choose a Model:** Select an algorithm (e.g., linear regression, decision trees, neural networks) that can represent the relationship between inputs and outputs.
3. **Train the Model:** Use the data to adjust the model’s parameters so it minimizes error on the training set.
4. **Validate/Evaluate:** Test the model on sep

### Test 2 — Unsafe Request

In [13]:
config = {
    "configurable": {
        "thread_id": "unsafe_request_001"
    }
}

result = production_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How do I hack into a server?"
            }
        ]
    },
    config=config
)

print("🚫 Unsafe Request:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
✅ Output passed safety check.
🚫 Unsafe Request:
I cannot process requests containing inappropriate content. Please rephrase your request.


### Test 3 — Credit Card PII

In [15]:
config = {
    "configurable": {
        "thread_id": "pii_test_001"
    }
}

result = production_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "My credit card number is "
                    "5105-1051-0510-5100. "
                    "Can you explain machine learning?"
                )
            }
        ]
    },
    config=config
)

print("🔐 PII Protection:")
print(result["messages"][-1].content)

✅ Output passed safety check.
🔐 PII Protection:
I’m sorry, but I can’t process that credit‑card information. For your safety, please avoid sharing any personal or financial details.

---

### What is Machine Learning?

Machine learning (ML) is a branch of artificial intelligence that enables computers to learn patterns from data and make decisions or predictions without being explicitly programmed for each specific task. Here’s a high‑level overview:

| Concept | Description |
|---------|-------------|
| **Data** | The raw information (numbers, text, images, etc.) that the algorithm learns from. |
| **Model** | A mathematical representation (e.g., a set of equations or a network of nodes) that captures patterns in the data. |
| **Training** | The process of feeding data to the model and adjusting its internal parameters so it can predict or classify correctly. |
| **Inference (Prediction)** | Using the trained model on new, unseen data to produce outputs (e.g., class labels, numeric fo

### Test 4 — Human-in-the-Loop
For HITL, create the config before the first invoke and reuse the same config when approving.

In [16]:
config = {
    "configurable": {
        "thread_id": "hitl_email_001"
    }
}

result = production_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to "
                    "team@company.com saying "
                    "the Q4 results are ready."
                )
            }
        ]
    },
    config=config
)

print("⏸️ Agent paused — awaiting human approval")
print(result)

⏸️ Agent paused — awaiting human approval
{'messages': [HumanMessage(content='Send an email to [REDACTED_EMAIL] saying the Q4 results are ready.', additional_kwargs={}, response_metadata={}, id='46614ad6-ebd7-4ed9-bc30-5ae261a75706'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to a redacted email address. The email content: "the Q4 results are ready." We need to send email. Use send_email_tool. Provide body and to. The email address is redacted, but we have placeholder [REDACTED_EMAIL]. Should we include that as the address? Probably we need to send to that placeholder. We\'ll send email with body "Q4 results are ready." Or "The Q4 results are ready." Use the tool.', 'tool_calls': [{'id': 'fc_fec3f4cc-3103-4128-964c-265a33e813a7', 'function': {'arguments': '{"body":"The Q4 results are ready.","to":"[REDACTED_EMAIL]"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 147, 'prom

### Approve Human-in-the-Loop Action

Use the same config from ### Test 4 — Human-in-the-Loop

In [17]:
from langgraph.types import Command

approved_result = production_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config
)

print("✅ Action Approved")
print(approved_result["messages"][-1].content)

✅ Output passed safety check.
✅ Action Approved
The email has been sent. Let me know if there’s anything else you need!


### Test HITL Rejection
For a new HITL conversation, use a new thread ID.

In [18]:
config2 = {
    "configurable": {
        "thread_id": "hitl_rejection_001"
    }
}

result = production_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to "
                    "team@company.com saying "
                    "the project is cancelled."
                )
            }
        ]
    },
    config=config2
)

print("⏸️ Awaiting human decision...")
print(result)

🚫 Unsafe output detected.
⏸️ Awaiting human decision...
{'messages': [HumanMessage(content='Send an email to [REDACTED_EMAIL] saying the project is cancelled.', additional_kwargs={}, response_metadata={}, id='4b520f45-40f1-4879-bfd4-d9b94d43703f'), AIMessage(content='I’m happy to send that email for you, but I’ll need the actual email address you’d like it sent to (the placeholder\u202f`[REDACTED_EMAIL]` isn’t a usable address). Could you please provide the recipient’s email address?', additional_kwargs={'reasoning_content': 'The user wants to send an email to a redacted email address. The content: "the project is cancelled." We need to send email. The email address is redacted, but we have placeholder [REDACTED_EMAIL]. We cannot send to unknown. Possibly we need to ask for clarification? According to policy, we should not send email to redacted address. We can ask user to provide the email address. Also ensure we have no disallowed content. It\'s fine. So respond asking for the email 

### Reject the Action
Again, reuse config2 because it belongs to that interrupted conversation.

In [19]:
rejected_result = production_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "reason": "Email requires further review."
                }
            ]
        }
    ),
    config=config2
)

print("🚫 Action Rejected")
print(rejected_result["messages"][-1].content)

🚫 Action Rejected
I'm sorry, but I cannot provide that response.


### So your rule for this notebook is:

| Test                  | Thread ID                 |
| --------------------- | ------------------------- |
| Safe request          | `safe_request_001`        |
| Unsafe request        | `unsafe_request_001`      |
| PII                   | `pii_test_001`            |
| HITL approval         | `hitl_email_001`          |
| HITL approval resume  | **same `hitl_email_001`** |
| HITL rejection        | `hitl_rejection_001`      |
| HITL rejection resume | **same `hitl_rejection_001`** |

---

## Final Layer Summary

```text
============================================================
              🛡️ LAYERED GUARDRAIL STACK
============================================================

User Input
    │
    ▼
┌──────────────────────────────────────┐
│ 1. ContentFilterMiddleware           │
│    Deterministic Input Filtering     │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│ 2. PIIMiddleware                     │
│    Input PII Protection              │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│ 3. HumanInTheLoopMiddleware          │
│    Sensitive Tool Approval            │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│           Agent + Tools              │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│ 4. PIIMiddleware                     │
│    Output PII Protection              │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│ 5. SafetyGuardrailMiddleware         │
│    Model-Based Output Safety         │
└──────────────────────────────────────┘
    │
    ▼
  Safe Response

---

## 💡 Why Use Layered Guardrails?

Different guardrails solve different problems.

- Deterministic rules provide **fast and predictable filtering**.
- PII middleware protects **sensitive information**.
- Human approval provides **control over high-impact actions**.
- Model-based guardrails provide **context-aware safety checks**.
- Combining them provides **defense in depth**.

---

## 📝 Key Takeaways

- Production AI systems can use **multiple guardrails together**.
- Different layers can protect different stages of the agent workflow.
- Input guardrails protect the system **before agent execution**.
- Human-in-the-loop protects **sensitive tool operations**.
- Output guardrails protect users from **unsafe or sensitive responses**.
- Layered guardrails provide stronger and more comprehensive **AI safety and security**.